# Preprocesamiento de Videos ASL - NSLT-100

Este notebook preprocesa los videos del dataset WLASL usando el subconjunto NSLT-100:
1. Carga metadatos
2. Identifica largest bounding box
3. Recorta y redimensiona a 224x224
4. Realiza muestreo temporal a 30 frames
5. Extrae características con MobileNetV2

## 1. Setup e Instalación de Dependencias

In [ ]:
# Si estás en Colab, monta Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("No se detectó Google Colab, ejecutando localmente")

In [ ]:
# Instalar dependencias (si es necesario)
!pip install opencv-python torch torchvision tqdm numpy

In [ ]:
import sys
import os

# Configurar paths
if IN_COLAB:
    # Ajustar a tu ruta en Google Drive
    PROJECT_PATH = '/content/drive/MyDrive/Kophos'
else:
    # Path local
    PROJECT_PATH = os.path.abspath('..')

print(f"Project path: {PROJECT_PATH}")
sys.path.insert(0, os.path.join(PROJECT_PATH, 'scripts'))

## 2. Importar Módulos y Verificar Datos

In [ ]:
from utils import (
    load_wlasl_metadata,
    load_nslt_subset,
    get_video_info,
    print_dataset_stats
)
from preprocess import VideoPreprocessor, FeatureExtractor, process_dataset

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Definir paths a los archivos
WLASL_JSON = os.path.join(PROJECT_PATH, 'data/raw/WLASL_v0.3.json')
NSLT_JSON = os.path.join(PROJECT_PATH, 'data/raw/nslt_100.json')
VIDEOS_DIR = os.path.join(PROJECT_PATH, 'data/raw/videos')  # Donde están tus videos descargados
OUTPUT_DIR = os.path.join(PROJECT_PATH, 'data')

# Verificar que los archivos existen
print(f"WLASL JSON exists: {os.path.exists(WLASL_JSON)}")
print(f"NSLT JSON exists: {os.path.exists(NSLT_JSON)}")
print(f"Videos directory exists: {os.path.exists(VIDEOS_DIR)}")

## 3. Cargar y Explorar Metadatos

In [ ]:
# Cargar metadatos
print("Cargando WLASL metadata...")
wlasl_data = load_wlasl_metadata(WLASL_JSON)
print(f"Total de glosses en WLASL: {len(wlasl_data)}")

print("\nCargando NSLT-100 subset...")
nslt_data = load_nslt_subset(NSLT_JSON)
print(f"Total de videos en NSLT-100: {len(nslt_data)}")

In [ ]:
# Ver estadísticas del dataset
print_dataset_stats(nslt_data)

## 4. Prueba con un Video Individual

In [ ]:
# Seleccionar un video de ejemplo
sample_video_id = list(nslt_data.keys())[0]
print(f"Video de ejemplo: {sample_video_id}")

# Obtener info completa
video_info = get_video_info(sample_video_id, wlasl_data, nslt_data)
print(f"\nInformación del video:")
for key, value in video_info.items():
    print(f"  {key}: {value}")

In [ ]:
# Procesar un video de prueba
preprocessor = VideoPreprocessor(target_size=(224, 224), target_frames=30)

sample_video_path = os.path.join(VIDEOS_DIR, f"{sample_video_id}.mp4")

if os.path.exists(sample_video_path):
    print("Procesando video de prueba...")
    processed_frames = preprocessor.preprocess_video(
        sample_video_path,
        video_info['bbox']
    )
    print(f"Shape de frames procesados: {processed_frames.shape}")
    
    # Visualizar algunos frames
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, ax in enumerate(axes.flat):
        frame_idx = i * 3  # Mostrar frames espaciados
        ax.imshow(processed_frames[frame_idx])
        ax.set_title(f"Frame {frame_idx}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Video no encontrado: {sample_video_path}")
    print("Nota: Necesitas descargar los videos primero")

## 5. Extracción de Características con MobileNetV2

In [ ]:
# Inicializar extractor de características
feature_extractor = FeatureExtractor()
print(f"Usando dispositivo: {feature_extractor.device}")

In [ ]:
# Extraer features del video de prueba
if os.path.exists(sample_video_path):
    print("Extrayendo características...")
    features = feature_extractor.extract_features(processed_frames)
    print(f"Shape de features: {features.shape}")
    print(f"Features extraídas: {features.shape[0]} frames x {features.shape[1]} dimensiones")

## 6. Procesamiento Completo del Dataset

**ADVERTENCIA**: Este paso procesará todos los videos. Asegúrate de tener:
- Suficiente espacio en disco
- Todos los videos descargados en `VIDEOS_DIR`
- GPU disponible (opcional, pero recomendado)

In [ ]:
# Procesar dataset completo
# NOTA: Esto puede tomar varias horas dependiendo de tu hardware

process_dataset(
    wlasl_json_path=WLASL_JSON,
    nslt_json_path=NSLT_JSON,
    videos_dir=VIDEOS_DIR,
    output_dir=OUTPUT_DIR,
    extract_features=True,    # Cambiar a False si solo quieres preprocesar frames
    save_frames=False         # Cambiar a True si quieres guardar frames (ocupa mucho espacio)
)

## 7. Verificar Resultados

In [ ]:
# Verificar archivos generados
features_dir = os.path.join(OUTPUT_DIR, 'features')

if os.path.exists(features_dir):
    feature_files = [f for f in os.listdir(features_dir) if f.endswith('.npy')]
    print(f"Archivos de features generados: {len(feature_files)}")
    
    # Cargar una feature de ejemplo
    if feature_files:
        sample_feature_file = os.path.join(features_dir, feature_files[0])
        sample_data = np.load(sample_feature_file, allow_pickle=True).item()
        
        print(f"\nEjemplo de datos guardados:")
        print(f"  Video ID: {sample_data['video_id']}")
        print(f"  Gloss: {sample_data['gloss']}")
        print(f"  Label: {sample_data['action_label']}")
        print(f"  Subset: {sample_data['subset']}")
        print(f"  Features shape: {sample_data['features'].shape}")
else:
    print("No se encontraron features. Ejecuta el procesamiento completo primero.")

## 8. Siguiente Paso: Entrenamiento

Los features están ahora listos para ser usados en el entrenamiento del modelo CNN+BiLSTM+Attention.

Estructura de los archivos generados:
```
data/
├── features/
│   ├── 05237.npy  # Cada archivo contiene features + metadata
│   ├── 69422.npy
│   └── ...
```

Formato de cada archivo .npy:
```python
{
    'features': np.array,      # Shape: (30, 1280) - 30 frames, 1280 features de MobileNetV2
    'video_id': str,           # ID del video
    'action_label': int,       # Índice de la clase (0-99)
    'subset': str,             # 'train', 'val', o 'test'
    'gloss': str               # Nombre de la señal
}
```